# Predicción de Precios Inmobiliarios — 2do Término
**UNAB · Inteligencia Artificial · Prof. Ph.D. Andrés Felipe Jerez Ariza**

## 1. Contexto y continuidad
- **Problema (igual al 1er término):** predecir el precio real de venta (`Sale Price`) de una propiedad a partir de sus características y su ubicación.
- **Dataset (el mismo):** Real Estate Property Transactions (Kaggle).
- **Tipo de aprendizaje:** supervisado — regresión, porque el objetivo es un número continuo y cada venta trae su precio real.
- **Resultado esperado:** un modelo con menor error que predecir siempre la mediana, y saber qué variables pesan más en el precio.

**Cambios respecto al 1er término:**
| Cambio | Por qué |
|---|---|
| Se agregan MSE y R² | La rúbrica los pide; R² dice qué % del precio explica el modelo |
| Se entrena con log(precio) | El precio está muy sesgado por las mansiones |
| Se agrega PCA | Lo pide el 2do término; se compara cada modelo con y sin PCA |
| No se usa `Estimated Value` | Es una tasación casi igual al precio: el modelo solo la copiaría |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib, json, os
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
sns.set_theme(style="whitegrid")

In [ ]:
# Cargar el CSV de Kaggle (en Colab se pide subirlo)
ARCHIVO = os.environ.get("DATA_PATH", "real_estate.csv")
if not os.path.exists(ARCHIVO):
    from google.colab import files
    ARCHIVO = next(iter(files.upload()))

df = pd.read_csv(ARCHIVO)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")  # 'Sale Price' -> 'sale_price'
print(df.shape)
df.head()

## 2. Limpieza de datos
Cada paso guarda cuántas filas o celdas afectó. En el 1er término reportamos: +5,000 registros, 2.5% de nulos en área y tasa de impuestos, fechas como texto y outliers de lujo.

In [ ]:
registro = []
def anotar(paso, n): registro.append({"Paso": paso, "Afectados": int(n)})
anotar("Filas iniciales", len(df))

# Tipos de datos: la fecha venía como texto
df["date"] = pd.to_datetime(df["date"], errors="coerce")
anotar("Fechas convertidas de texto a fecha", df["date"].notna().sum())

# Formatos inconsistentes: espacios y mayúsculas en las categorías
for c in ["locality", "property", "residential", "face"]:
    antes = df[c].copy()
    df[c] = df[c].str.strip().str.title()
    anotar(f"'{c}' con formato corregido", (antes.notna() & (antes != df[c])).sum())

# Errores de digitación en categorías: "?" no es un valor válido
for c in ["locality", "property", "residential", "face"]:
    n = (df[c] == "?").sum()
    if n: anotar(f"'{c}' con '?' -> faltante", n)
    df[c] = df[c].replace("?", np.nan)

# Localidad faltante: se deja como categoría propia (rellenarla con la más común sesgaría el precio)
anotar("Localidad faltante -> 'Desconocida'", df["locality"].isna().sum())
df["locality"] = df["locality"].fillna("Desconocida")

# Duplicados
anotar("Duplicados eliminados", df.duplicated().sum())
df = df.drop_duplicates()

# Errores de digitación: precio, área o habitaciones imposibles
malos = (df["sale_price"] <= 0) | (df["carpet_area"] <= 0) | (df["num_rooms"] <= 0)
anotar("Valores imposibles eliminados", malos.sum())
df = df[~malos]

# Faltantes: se rellenan con la mediana dentro del modelo (solo con datos de entrenamiento)
for c in ["carpet_area", "property_tax_rate"]:
    anotar(f"Nulos en '{c}' (se imputan con mediana)", df[c].isna().sum())

# Outliers: se eliminan solo los extremos (> 3 IQR sobre log del precio); las casas de lujo se quedan
lp = np.log(df["sale_price"])
q1, q3 = lp.quantile([.25, .75])
extremos = (lp < q1 - 3 * (q3 - q1)) | (lp > q3 + 3 * (q3 - q1))
anotar("Outliers extremos eliminados", extremos.sum())
df = df[~extremos].reset_index(drop=True)

anotar("Filas finales", len(df))
pd.DataFrame(registro)

## 3. Análisis exploratorio (EDA)

In [ ]:
# Univariado: el precio está sesgado; con log queda simétrico
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["sale_price"], bins=50, ax=ax[0]).set_title("Sale Price")
sns.histplot(np.log(df["sale_price"]), bins=50, ax=ax[1]).set_title("log(Sale Price)")
plt.show()
print("Asimetría del precio:", round(df["sale_price"].skew(), 2), "-> con log:", round(np.log(df["sale_price"]).skew(), 2))

In [ ]:
# Bivariado: área vs precio y precio por localidad
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="carpet_area", y="sale_price", alpha=0.3, ax=ax[0]).set_title("Área vs precio")
orden = df.groupby("locality")["sale_price"].median().sort_values().index
sns.boxplot(data=df, y="locality", x="sale_price", order=orden, showfliers=False, ax=ax[1]).set_title("Precio por localidad")
plt.tight_layout(); plt.show()
print("Correlación área-precio:", round(df["carpet_area"].corr(df["sale_price"], method="spearman"), 2))

In [ ]:
# Multivariado: correlaciones y localidad x habitaciones
num = ["sale_price", "estimated_value", "carpet_area", "num_rooms", "num_bathrooms", "property_tax_rate"]
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(df[num].corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax[0])
tabla = df.pivot_table(index="locality", columns="num_rooms", values="sale_price", aggfunc="median") / 1000
sns.heatmap(tabla, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax[1]).set_title("Precio mediano (miles)")
plt.tight_layout(); plt.show()

**Insights:**
- **Univariado:** el precio es muy asimétrico (asimetría ≈ 6.5; con log ≈ −0.1) → se entrena con log(precio). El 87% de las propiedades tiene 3 habitaciones.
- **Bivariado:** a diferencia de lo que planteamos en el 1er término, el área tiene correlación **débil** con el precio (≈ −0.2). Lo que más cambia el precio es la **localidad**: la mediana de Greenwich es unas 15 veces la de Waterbury.
- **Multivariado:** `estimated_value` tiene correlación ≈ 0.9 con el precio (por eso no se usa: el modelo solo copiaría la tasación). El tipo residencial y las habitaciones van juntos (más habitaciones = dúplex/tríplex, más baratos).
- **Para el modelo:** la localidad y el tipo de propiedad son las variables clave; como la relación no es lineal, se esperan mejores resultados con modelos de árboles.

## 4. Feature engineering
- `sale_year`, `sale_month`: el mercado cambia con el tiempo.
- `area_per_room`: área ÷ habitaciones (qué tan amplios son los espacios).
- Numéricas: mediana + `RobustScaler` (resistente a outliers). Categóricas: moda + One-Hot.

In [ ]:
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["area_per_room"] = df["carpet_area"] / df["num_rooms"]

NUMERICAS = ["carpet_area", "num_rooms", "num_bathrooms", "property_tax_rate", "sale_year", "sale_month", "area_per_room"]
CATEGORICAS = ["locality", "property", "residential", "face"]
X, y = df[NUMERICAS + CATEGORICAS], df["sale_price"]

preproceso = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", RobustScaler())]), NUMERICAS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAS)])

# Separación temporal: el 20% de ventas más recientes es el test
orden = df["date"].sort_values().index
corte = int(len(orden) * 0.8)
X_train, X_test = X.loc[orden[:corte]], X.loc[orden[corte:]]
y_train, y_test = y.loc[orden[:corte]], y.loc[orden[corte:]]
print("Train:", len(X_train), "| Test:", len(X_test))

## 5. Reducción con PCA
El One-Hot de localidades crea muchas columnas y el área, las habitaciones y los baños están correlacionados. Se usa PCA (no LDA, porque LDA es para clasificación) y se eligen los componentes que explican el 95% de la varianza.

In [ ]:
Z = preproceso.fit_transform(X_train)
pca = PCA().fit(Z)
acumulada = np.cumsum(pca.explained_variance_ratio_)
N_COMP = int(np.argmax(acumulada >= 0.95) + 1)

plt.plot(range(1, len(acumulada) + 1), acumulada, marker=".")
plt.axhline(0.95, color="red", ls="--"); plt.axvline(N_COMP, color="green", ls=":")
plt.xlabel("Componentes"); plt.ylabel("Varianza acumulada"); plt.title("PCA"); plt.show()
print(f"{Z.shape[1]} columnas -> {N_COMP} componentes ({acumulada[N_COMP-1]*100:.1f}% de la varianza)")

## 6. Modelado
- **Modelos:** Regresión Lineal (base simple), Random Forest (capta interacciones), XGBoost (suele ser el más preciso en tablas). Cada uno **con y sin PCA**.
- **Validación:** test temporal (80/20) + K-Fold de 5 sobre el train.
- **Métricas:** RMSE (principal, castiga errores grandes), MAE (error promedio en dinero), MSE y R².

In [ ]:
MODELOS = {
    "Regresión Lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=42),
}

def armar(modelo, con_pca):
    pasos = [("prep", preproceso)] + ([("pca", PCA(n_components=N_COMP))] if con_pca else []) + [("modelo", modelo)]
    # Entrena con log(precio) y devuelve el precio normal
    return TransformedTargetRegressor(regressor=Pipeline(pasos), func=np.log1p, inverse_func=np.expm1)

filas, entrenados = [], {}
for nombre, modelo in MODELOS.items():
    for con_pca in [False, True]:
        etiqueta = nombre + (" + PCA" if con_pca else "")
        m = armar(modelo, con_pca)
        cv = -cross_val_score(m, X_train, y_train, cv=KFold(5, shuffle=True, random_state=42),
                              scoring="neg_root_mean_squared_error")
        m.fit(X_train, y_train)
        p = m.predict(X_test)
        filas.append({"Modelo": etiqueta, "CV RMSE": cv.mean(),
                      "RMSE": np.sqrt(mean_squared_error(y_test, p)), "MAE": mean_absolute_error(y_test, p),
                      "MSE": mean_squared_error(y_test, p), "R2 test": r2_score(y_test, p),
                      "R2 train": r2_score(y_train, m.predict(X_train))})
        entrenados[etiqueta] = m

res = pd.DataFrame(filas).set_index("Modelo").sort_values("RMSE")
res["Diagnóstico"] = np.where(res["R2 train"] < 0.6, "Underfitting",
                     np.where(res["R2 train"] - res["R2 test"] > 0.10, "Overfitting", "Bien"))
res.round(3)

## 7. Resultados

In [ ]:
MEJOR = res[res["Diagnóstico"] == "Bien"].index[0] if (res["Diagnóstico"] == "Bien").any() else res.index[0]
mejor = entrenados[MEJOR]
r = res.loc[MEJOR]
print(f"Mejor modelo: {MEJOR}")
print(f"RMSE {r['RMSE']:,.0f} | MAE {r['MAE']:,.0f} | R² {r['R2 test']:.3f}")

base = np.full(len(y_test), y_train.median())
print(f"Predecir siempre la mediana: RMSE {np.sqrt(mean_squared_error(y_test, base)):,.0f}")

p = mejor.predict(X_test)
plt.scatter(y_test, p, alpha=0.3); plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Precio real"); plt.ylabel("Precio predicho"); plt.title(MEJOR); plt.show()

In [ ]:
# Importancia de variables (Random Forest sin PCA, para poder interpretarlas)
rf = entrenados["Random Forest"].regressor_
imp = pd.Series(rf.named_steps["modelo"].feature_importances_, index=rf.named_steps["prep"].get_feature_names_out())
imp.sort_values().tail(10).plot.barh(title="Variables más importantes"); plt.show()

## 8. Conclusiones y limitaciones
**Conclusiones (completen con sus números):**
- El mejor modelo es **[__]**: explica el **[__]%** de la variación del precio en ventas que no vio, con un error promedio de **$[__]**.
- Supera claramente a predecir siempre la mediana, así que sí se puede estimar el precio con características de la propiedad.
- Las variables que más pesan son **[__]** (se espera área y localidad).
- PCA **[ayudó / no ayudó]**: comparen las filas con y sin PCA de la tabla.

**Limitaciones:** faltan variables (antigüedad, parqueadero, estado), hay pocas casas de lujo y el mercado cambia con el tiempo.
**Mejoras:** más variables, más datos recientes y re-entrenar el modelo cada cierto tiempo.

## 9. Exportar el modelo para la app

In [ ]:
final = armar(MODELOS[MEJOR.replace(" + PCA", "")], "PCA" in MEJOR).fit(X, y)  # se re-entrena con todos los datos
joblib.dump(final, "modelo.joblib")
info = {"modelo": MEJOR, "rmse": r["RMSE"], "mae": r["MAE"], "r2": r["R2 test"],
        "opciones": {c: sorted(df[c].dropna().unique().tolist()) for c in CATEGORICAS},
        "ultima_fecha": str(df["date"].max().date()),
        "mediana_localidad": df.groupby("locality")["sale_price"].median().to_dict()}
json.dump(info, open("info.json", "w"), ensure_ascii=False, indent=1)

import sklearn, xgboost
print("Pongan esto en requirements.txt:")
print(f"scikit-learn=={sklearn.__version__}\nxgboost=={xgboost.__version__}")

try:
    from google.colab import files
    files.download("modelo.joblib"); files.download("info.json")
except ImportError:
    pass